# ReleaseEvent pipeline test (`09_release_events`)

This notebook tests `09_release_events` with a small synthetic Bronze table in the schema `github_lakehouse.test_release`. It runs the real notebook with `%run` and `test_schema = test_release`, so **production tables are never written**. The last check section proves that by comparing the production table versions before and after.

**Before you run it**, the production tables must exist, because the test tables are created with `CREATE TABLE ... LIKE`:

1. `11_silver_audit_table` (audit table; it already exists from the PushEvent work)
2. cells 1–6 of `09_release_events` (they create `silver.release_events` and `quarantine.quarantine_release_event`)

Every check prints `PASS` or `FAIL`. A section stops with an error if any of its checks failed. The last cell drops the test schema.

The shared pipeline (parse errors, core IDs, duplicates, all-dates mode, partial failure) is already covered by `tests/test_08_fork_events`. This test covers what is specific to ReleaseEvent: the six release fields are read and converted, `release_published_at` is never filled from `event_timestamp`, an unknown draft or prerelease flag stays `NULL` (never `false`), every problem in a release field is kept with a warning instead of quarantined, an action other than `published` is kept with an `UNEXPECTED_VALUE` warning, and the quarantine table keeps the release fields and their original text.

## B1. Setup: test schema, empty tables and helpers

In [0]:
import json
from datetime import date, datetime, timezone

TEST_SCHEMA = "test_release"
assert TEST_SCHEMA.startswith("test_")   # never drop anything outside a test schema

def T(name):
    return f"github_lakehouse.{TEST_SCHEMA}.{name}"

TEST_BRONZE = T("github_events_raw")
TEST_SILVER = T("release_events")
TEST_QUARANTINE = T("quarantine_release_event")
TEST_AUDIT = T("silver_pipeline_runs")

PROD_TABLES = [
    "github_lakehouse.bronze.github_events_raw",
    "github_lakehouse.silver.release_events",
    "github_lakehouse.quarantine.quarantine_release_event",
    "github_lakehouse.audit.silver_pipeline_runs",
]

# The test tables copy the production definitions, so those must exist first.
missing = [t for t in PROD_TABLES if not spark.catalog.tableExists(t)]
if missing:
    raise RuntimeError(
        f"Missing production tables: {missing}. Run 11_silver_audit_table "
        "and cells 1-6 of 09_release_events (table creation) first."
    )

spark.sql(f"CREATE SCHEMA IF NOT EXISTS github_lakehouse.{TEST_SCHEMA}")
for t in [TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT]:
    spark.sql(f"DROP TABLE IF EXISTS {t}")

# Bronze: only the four columns the pipeline reads.
spark.sql(f"""
    CREATE TABLE {TEST_BRONZE} (
        raw_json STRING, _source_file STRING,
        _source_date DATE, _ingested_at TIMESTAMP
    ) USING DELTA
""")
# Targets: exact copies of the production table definitions (no data).
spark.sql(f"CREATE TABLE {TEST_SILVER} LIKE github_lakehouse.silver.release_events")
spark.sql(f"CREATE TABLE {TEST_QUARANTINE} LIKE github_lakehouse.quarantine.quarantine_release_event")
spark.sql(f"CREATE TABLE {TEST_AUDIT} LIKE github_lakehouse.audit.silver_pipeline_runs")


# ---- assertion helpers ----
_results = []

def check(label, actual, expected):
    ok = actual == expected
    _results.append((label, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {label}: actual={actual!r} expected={expected!r}")

def finish(section):
    failed = [label for label, ok in _results if not ok]
    _results.clear()
    if failed:
        raise AssertionError(f"{section}: {len(failed)} check(s) failed: {failed}")
    print(f"\n{section}: ALL CHECKS PASSED")

def check_audit(expected_counts, expected_quality=None,
                expected_status="SUCCEEDED", reconciled=True):
    # `audit` is the dict created by the pipeline's audit cell via %run.
    check("audit status", audit["status"], expected_status)
    check("audit reconciliation_passed", audit["reconciliation_passed"], reconciled)
    for k, v in expected_counts.items():
        check(f"audit {k}", audit.get(k), v)
    for k, v in (expected_quality or {}).items():
        check(f"quality {k}", audit["quality_metrics"].get(k), v)

def rows(table, where="true"):
    return [r.asDict() for r in spark.table(table).where(where).collect()]

def one_silver(event_id):
    found = rows(TEST_SILVER, f"event_id = '{event_id}'")
    check(f"silver has exactly one '{event_id}'", len(found), 1)
    return found[0] if found else {}

def table_version(t):
    return spark.sql(f"DESCRIBE HISTORY {t} LIMIT 1").collect()[0]["version"]

print("Setup complete:", TEST_BRONZE, TEST_SILVER, TEST_QUARANTINE, TEST_AUDIT)

## B2. Safety snapshot of the production tables

In [0]:
prod_versions_before = {t: table_version(t) for t in PROD_TABLES}
prod_versions_before

## B3. Load the synthetic Bronze data

The default release is `id 5001`, tag `v1.0.0`, published `2024-02-01T09:30:00Z`, `draft = false`, `prerelease = false`, action `published`.

| Row | Case | Expected result |
|---|---|---|
| `r1` | clean ReleaseEvent | Silver, every field converted, no warning |
| `r2` | `published_at` missing | Silver, `release_published_at` NULL (not the event time), warning `MISSING: release_published_at` |
| `r3` | `published_at = "not-a-date"` | Silver, NULL, warning `INVALID_CAST: release_published_at` |
| `r4` | `draft` missing | Silver, `release_draft` NULL (not false), warning `MISSING: release_draft` |
| `r5` | `prerelease = 1` (a number) | Silver, NULL, warning `INVALID_CAST: release_prerelease` |
| `r6` | `tag_name` is blank | Silver, `release_tag` NULL, warning `MISSING: release_tag` |
| `r7` | `action = created` | Silver, value kept, warning `UNEXPECTED_VALUE: release_action` |
| `r8` | `id = "abc"` | Silver, `release_id` NULL, warning `INVALID_CAST: release_id` |
| `r9` | `draft = true`, `prerelease = true` | Silver, both true, no warning |
| `r10` | no `release` object | Silver, five `MISSING` warnings, one per release field |
| `r11` | `release` is text, not an object | quarantine `JSON_PARSE_ERROR` |
| `r12` | `repo.id = "x"` | quarantine `INVALID_CAST: repo_id`, release fields and original text kept |
| `r13` (twice) | duplicate `event_id`, older and newer ingestion | one Silver row, from the newer file |
| `x1` | a WatchEvent | ignored (not read by this notebook) |

In [0]:
F1 = "synthetic/2024-02-01-0.json.gz"
F2 = "synthetic/2024-02-01-1.json.gz"
D1 = date(2024, 2, 1)
I1 = datetime(2024, 3, 1, tzinfo=timezone.utc)   # earlier ingestion
I2 = datetime(2024, 3, 2, tzinfo=timezone.utc)   # later ingestion

def release(drop=(), **changes):
    """A release object. Defaults = a clean, published, non-draft release."""
    r = {"id": 5001, "tag_name": "v1.0.0", "published_at": "2024-02-01T09:30:00Z",
         "draft": False, "prerelease": False}
    r.update(changes)
    for key in drop:
        r.pop(key)
    return r

def ev(event_id, type_="ReleaseEvent", created_at="2024-02-01T10:00:00Z",
       actor=None, repo=None, action="published", rel=None, payload=None):
    """One GitHub-style event as a JSON string. Defaults = a clean ReleaseEvent."""
    event = {
        "id": event_id,
        "type": type_,
        "created_at": created_at,
        "actor": {"id": 101, "login": "alice"} if actor is None else actor,
        "repo": {"id": 201, "name": "octo/repo"} if repo is None else repo,
        "payload": ({"action": action, "release": release() if rel is None else rel}
                    if payload is None else payload),
    }
    return json.dumps(event)

synthetic_rows = [
    # (raw_json, _source_file, _source_date, _ingested_at)                  # case
    (ev("r1"),                                                F1, D1, I1),   # clean
    (ev("r2", rel=release(drop=["published_at"])),            F1, D1, I1),   # published_at missing
    (ev("r3", rel=release(published_at="not-a-date")),        F1, D1, I1),   # published_at invalid
    (ev("r4", rel=release(drop=["draft"])),                   F1, D1, I1),   # draft missing -> NULL
    (ev("r5", rel=release(prerelease=1)),                     F1, D1, I1),   # prerelease 1 -> NULL
    (ev("r6", rel=release(tag_name="  ")),                    F1, D1, I1),   # blank tag -> NULL
    (ev("r7", action="created"),                              F1, D1, I1),   # unexpected action
    (ev("r8", rel=release(id="abc")),                         F1, D1, I1),   # release_id invalid
    (ev("r9", rel=release(draft=True, prerelease=True)),      F1, D1, I1),   # true flags
    (ev("r10", payload={"action": "published"}),              F1, D1, I1),   # no release object
    (ev("r11", rel="oops"),                                   F1, D1, I1),   # parse error
    (ev("r12", repo={"id": "x", "name": "octo/repo"}),        F1, D1, I1),   # repo_id invalid
    (ev("r13", rel=release(id=1301)),                         F1, D1, I1),   # duplicate, older
    (ev("r13", rel=release(id=1302)),                         F2, D1, I2),   # duplicate, newer -> wins
    (ev("x1", type_="WatchEvent", payload={"action": "started"}),
                                                              F1, D1, I1),   # other event type
]

(spark.createDataFrame(synthetic_rows, spark.table(TEST_BRONZE).schema)
    .write.mode("overwrite").saveAsTable(TEST_BRONZE))

check("synthetic bronze rows", spark.table(TEST_BRONZE).count(), 15)
finish("B3 load")

## B4. Run 1: single date 2024-02-01

In [0]:
%run ../09_release_events $test_schema="test_release" $source_date="2024-02-01" $run_id="test-run1"

## B5. Run 1 checks

In [0]:
QUALITY_RUN1 = {
    "null_event_timestamp": 0, "null_actor_id": 0, "null_actor_login": 0,
    "null_repo_name": 0,
    "null_release_action": 0, "null_release_id": 2, "null_release_tag": 2,
    "null_release_published_at": 3, "null_release_draft": 2,
    "null_release_prerelease": 2, "unexpected_release_action": 1,
    "missing_lineage": 0,
}

# --- tables and audit identity: the run used the test tables only
check("resolved silver", SILVER_TABLE, TEST_SILVER)
check("resolved quarantine", QUARANTINE_TABLE, TEST_QUARANTINE)
check("audit pipeline_name", audit["pipeline_name"], "silver_release_events")
check("audit event_type", audit["event_type"], "ReleaseEvent")

# --- audit: 14 ReleaseEvent rows (x1 is a WatchEvent)
check_audit({
    "bronze_input_rows": 14, "hard_null_quarantine_rows": 2,
    "validation_quarantine_rows": 0, "validated_rows": 12,
    "duplicate_rows_removed": 1, "silver_candidate_rows": 11,
    "silver_rows_inserted": 11, "silver_rows_already_present": 0,
    "quarantine_candidate_rows": 2, "quarantine_duplicates_removed": 0,
    "quarantine_rows_inserted": 2,
}, QUALITY_RUN1)
check("quality metric names", sorted(audit["quality_metrics"]), sorted(QUALITY_RUN1))

# --- Silver schema: the contract columns, release fields before lineage
check("silver columns", spark.table(TEST_SILVER).columns, [
    "event_id", "event_type", "event_timestamp", "event_date",
    "actor_id", "actor_login", "repo_id", "repo_name",
    "release_action", "release_id", "release_tag", "release_published_at",
    "release_draft", "release_prerelease",
    "_source_file", "_source_date", "_ingested_at", "quarantine_warning",
])

# --- Silver contents
check("silver event_ids",
      sorted(r["event_id"] for r in rows(TEST_SILVER)),
      sorted(["r1", "r2", "r3", "r4", "r5", "r6", "r7", "r8", "r9", "r10", "r13"]))

def release_fields(r):
    return (r.get("release_action"), r.get("release_id"), r.get("release_tag"),
            r.get("release_draft"), r.get("release_prerelease"))

def published_text(event_id):
    found = (spark.table(TEST_SILVER).where(f"event_id = '{event_id}'")
             .selectExpr("date_format(release_published_at, 'yyyy-MM-dd HH:mm:ss') AS p")
             .collect())
    return found[0]["p"] if found else "row not found"

r = one_silver("r1")
check("r1 release fields", release_fields(r), ("published", 5001, "v1.0.0", False, False))
check("r1 published_at (UTC)", published_text("r1"), "2024-02-01 09:30:00")
check("r1 warning", r.get("quarantine_warning"), None)

r = one_silver("r2")
check("r2 published_at NULL, not the event time", published_text("r2"), None)
check("r2 event_timestamp still set", r.get("event_timestamp") is not None, True)
check("r2 warning", r.get("quarantine_warning"), "MISSING: release_published_at")

r = one_silver("r3")
check("r3 published_at", published_text("r3"), None)
check("r3 warning", r.get("quarantine_warning"), "INVALID_CAST: release_published_at")

r = one_silver("r4")
check("r4 draft is NULL, not False", r.get("release_draft"), None)
check("r4 prerelease still known", r.get("release_prerelease"), False)
check("r4 warning", r.get("quarantine_warning"), "MISSING: release_draft")

r = one_silver("r5")
check("r5 prerelease is NULL", r.get("release_prerelease"), None)
check("r5 warning", r.get("quarantine_warning"), "INVALID_CAST: release_prerelease")

r = one_silver("r6")
check("r6 tag", r.get("release_tag"), None)
check("r6 warning", r.get("quarantine_warning"), "MISSING: release_tag")

r = one_silver("r7")
check("r7 action kept unchanged", r.get("release_action"), "created")
check("r7 warning", r.get("quarantine_warning"), "UNEXPECTED_VALUE: release_action")

r = one_silver("r8")
check("r8 release_id", r.get("release_id"), None)
check("r8 warning", r.get("quarantine_warning"), "INVALID_CAST: release_id")

r = one_silver("r9")
check("r9 flags", (r.get("release_draft"), r.get("release_prerelease")), (True, True))
check("r9 warning", r.get("quarantine_warning"), None)

r = one_silver("r10")
check("r10 release fields", release_fields(r), ("published", None, None, None, None))
check("r10 warnings", r.get("quarantine_warning"),
      "MISSING: release_id; MISSING: release_tag; MISSING: release_published_at; "
      "MISSING: release_draft; MISSING: release_prerelease")

r = one_silver("r13")
check("r13 winner = newer ingestion", (r.get("release_id"), r.get("_source_file")), (1302, F2))

# --- Quarantine contents
q = rows(TEST_QUARANTINE)
check("quarantine reasons",
      sorted(r["quarantine_reason"] for r in q), ["INVALID_CAST: repo_id", "JSON_PARSE_ERROR"])
# A parse-error row is identified by its raw_json, because its parsed fields
# (including event_id) may be NULL.
parse_error = [r for r in q if r["quarantine_reason"] == "JSON_PARSE_ERROR"]
check("r11 is the parse error",
      [json.loads(r["raw_json"])["id"] for r in parse_error], ["r11"])
r12 = [r for r in q if r["event_id"] == "r12"]
check("r12 found once", len(r12), 1)
r = r12[0] if r12 else {}
check("r12 reason", r.get("quarantine_reason"), "INVALID_CAST: repo_id")
check("r12 release fields kept", release_fields(r), ("published", 5001, "v1.0.0", False, False))
check("r12 original text kept",
      (r.get("release_id_raw"), r.get("release_published_at_raw"),
       r.get("release_draft_raw"), r.get("release_prerelease_raw")),
      ("5001", "2024-02-01T09:30:00Z", "false", "false"))
check("quarantine keeps raw_json", all(r["raw_json"] for r in q), True)
check("quarantine keys non-NULL", all(r["quarantine_key"] for r in q), True)

finish("B5 run 1")

## B6. Run 2: rerun the same date

In [0]:
%run ../09_release_events $test_schema="test_release" $source_date="2024-02-01" $run_id="test-run2"

## B7. Run 2 checks: idempotency, plus the final safety check

In [0]:
check_audit({
    "bronze_input_rows": 14, "silver_candidate_rows": 11,
    "silver_rows_inserted": 0, "silver_rows_already_present": 11,
    "quarantine_candidate_rows": 2, "quarantine_rows_inserted": 0,
}, QUALITY_RUN1)
check("silver rows unchanged", spark.table(TEST_SILVER).count(), 11)
check("quarantine rows unchanged", spark.table(TEST_QUARANTINE).count(), 2)

statuses = [r["status"] for r in
            spark.table(TEST_AUDIT).orderBy("run_finished_at").collect()]
check("audit history", statuses, ["SUCCEEDED", "SUCCEEDED"])

for t in PROD_TABLES:
    check(f"prod version unchanged: {t}", table_version(t), prod_versions_before[t])
finish("B7 run 2 + production safety")

## Clean up: drop the test schema

In [0]:
assert TEST_SCHEMA.startswith("test_")
spark.sql(f"DROP SCHEMA IF EXISTS github_lakehouse.{TEST_SCHEMA} CASCADE")